# 🎯 Command — Combined Routing + State Updates

## Learning Objectives
In this notebook, you will learn:
1. **`Command` object** — updating state and routing in a single return value
2. **Type annotations** — using `Command[Literal[...]]` for compile-time safety
3. **No explicit edges needed** — `Command.goto` handles routing automatically

## Prerequisites
- `langgraph` installed
- Understanding of conditional edges (notebook `08-edges`)

---
## 🔧 Part 1: Environment Setup

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: Imports
# ============================================================================
from typing import TypedDict, Literal

from langgraph.graph import StateGraph, START, END
from langgraph.types import Command

print("✅ Imports loaded successfully!")

✅ Imports loaded successfully!


---
## 📋 Part 2: Define the State

In [2]:
# ============================================================================
# STATE DEFINITION
# ============================================================================
class GraphState(TypedDict):
    temperature: int
    status_messages: str
    warning_sent: bool
    final_action_performed: str

print("✅ State defined!")

✅ State defined!


---
## ⚙️ Part 3: Define the Command-Enabled Node

Instead of returning a plain dict and using a separate conditional edge,
a `Command`-enabled node does **both** in one return:
- `update` — partial state update (like a normal node return)
- `goto` — the name of the next node to execute

> **Key Insight**: The return type annotation `Command[Literal["warn_user", "success"]]`
> tells LangGraph at compile time which nodes this function can route to.

In [3]:
# ============================================================================
# COMMAND NODE: Route and update state simultaneously
# ============================================================================
def check_temp_node(state: GraphState) -> Command[Literal["warn_user", "success"]]:
    """
    Uses Command to both update state and route conditionally.
    The return type must be annotated with Command and a Literal list
    of all possible destination nodes.
    """
    temp = state["temperature"]
    print(f"Executing 'check_temp_node': Current Temp is {temp} degrees")

    if temp > 90:
        print("--- ALERT: Temp too high! Issuing command to warn user ---")
        return Command(
            update={
                "status_messages": "Routing to warning handler...",
                "warning_sent": True
            },
            goto="warn_user"
        )
    else:
        print("--- OK: Temp is safe. Routing to 'success' ---")
        return Command(
            update={
                "status_messages": "Routing to success handler..."
            },
            goto="success"
        )

print("✅ Command node defined!")

✅ Command node defined!


---
## 🔔 Part 4: Define the Action Nodes

In [4]:
# ============================================================================
# ACTION NODES: warn_user and success
# ============================================================================
def warn_user(state: GraphState):
    print(f"Executing 'warn_user': Warning sent at {state['temperature']}°C.")
    return {"final_action_performed": "Warning Notification sent"}


def success(state: GraphState):
    print("Executing 'success': Process completed successfully.")
    return {"final_action_performed": "Temperature Safety Confirmed"}

print("✅ Action nodes defined!")

✅ Action nodes defined!


---
## 🔗 Part 5: Build the Graph

Notice that we **don't** add edges from `check_temp_node` to `warn_user`
or `success` — the `Command.goto` handles routing. We also don't need
explicit edges to `END` from the action nodes.

In [5]:
# ============================================================================
# GRAPH CONSTRUCTION: Command-based routing
# ============================================================================
builder = StateGraph(GraphState)

builder.add_node(check_temp_node)
builder.add_node(warn_user)
builder.add_node(success)

builder.add_edge(START, "check_temp_node")

graph = builder.compile()

print("✅ Graph compiled!")

✅ Graph compiled!


---
## 🚀 Part 6: Test Both Paths

### Test 1: High Temperature (triggers warning)

In [6]:
# ============================================================================
# TEST 1: High temperature — routes to warn_user
# ============================================================================
final_state_1 = graph.invoke({"temperature": 100})

print("\n🔥 High Temperature Final State:")
print(final_state_1)
print("=" * 50)

Executing 'check_temp_node': Current Temp is 100 degrees
--- ALERT: Temp too high! Issuing command to warn user ---
Executing 'warn_user': Warning sent at 100°C.

🔥 High Temperature Final State:
{'temperature': 100, 'status_messages': 'Routing to warning handler...', 'warning_sent': True, 'final_action_performed': 'Warning Notification sent'}


### Test 2: Low Temperature (success path)

In [7]:
# ============================================================================
# TEST 2: Low temperature — routes to success
# ============================================================================
final_state_2 = graph.invoke({"temperature": 40})

print("\n✅ Low Temperature Final State:")
print(final_state_2)

Executing 'check_temp_node': Current Temp is 40 degrees
--- OK: Temp is safe. Routing to 'success' ---
Executing 'success': Process completed successfully.

✅ Low Temperature Final State:
{'temperature': 40, 'status_messages': 'Routing to success handler...', 'final_action_performed': 'Temperature Safety Confirmed'}


---
## 📝 Summary

In this notebook, we learned:

### 1. `Command` Object
- Combines **state update** (`update=`) and **routing** (`goto=`) in one return
- Replaces the need for separate conditional edge functions

### 2. Type Safety with `Literal`
- `Command[Literal["node_a", "node_b"]]` declares valid destinations at compile time

### 3. Simplified Graph Wiring
- No need for explicit `add_edge` or `add_conditional_edges` from Command nodes
- Destination nodes auto-terminate at `END`

### Next Steps
- Explore **Retries** for fault-tolerant nodes (notebook `12-retries`)